ElastiCache is AWS's managed in-memory caching service. It reduces database load, cuts response times from milliseconds to microseconds, and handles session storage and real-time leaderboards at scale. ElastiCache supports two engines: Redis and Memcached — each with distinct trade-offs that the SAA-C03 exam tests directly.

## Why In-Memory Caching?

```text
Without cache:                    With cache:
App → DB (10ms per query)         App → Cache (sub-ms, cache HIT)
  ↑ repeated identical queries      ↓ cache MISS only
  DB gets hammered                 App → DB (10ms, result cached)
```

### Common caching patterns

**Lazy loading (cache-aside):**
1. App checks cache
2. Cache HIT → return data
3. Cache MISS → fetch from DB → write to cache → return data
- Pros: Only caches data that is actually requested; cache failure is not fatal
- Cons: First request is always slow (cache miss); data can be stale

**Write-through:**
1. App writes to DB
2. App (or DB trigger) also writes to cache
- Pros: Cache is always fresh
- Cons: Write penalty (every write hits both DB and cache); cache may fill with data never read

**Session store:**
Store user session data in ElastiCache so any application server can retrieve it — enables stateless, horizontally scalable app tiers.

## Redis vs Memcached

| Feature | Redis | Memcached |
|---|---|---|
| **Data structures** | Strings, Hashes, Lists, Sets, Sorted Sets, Bitmaps, HyperLogLog, Streams | Strings only |
| **Persistence** | Yes — RDB snapshots + AOF (append-only file) | No |
| **Replication** | Yes — primary + read replicas | No |
| **Multi-AZ failover** | Yes — automatic with Redis Cluster or Replication Group | No |
| **Pub/Sub messaging** | Yes | No |
| **Lua scripting** | Yes | No |
| **Transactions** | Yes (MULTI/EXEC) | No |
| **Cluster mode** | Yes (sharding across nodes) | Yes (horizontal scaling) |
| **Multi-threading** | Single-threaded (mostly) | Multi-threaded |
| **Backup/restore** | Yes | No |

> **Rule of thumb:** If you need anything beyond simple key-value caching — persistence, replication, complex data types, pub/sub, sorted sets — use Redis. Use Memcached only for simple, horizontally scaled, throwaway caching where multi-threading matters.

> **Exam tip:** Almost every exam scenario that mentions ElastiCache is pointing at Redis. Memcached appears rarely and only when the scenario emphasises simplicity or multi-threading.

## ElastiCache for Redis — Deep Dive

### Replication Group
A **Replication Group** is a cluster of Redis nodes:
- 1 **primary** node (reads + writes)
- Up to 5 **read replicas** per shard
- Multi-AZ: replicas in different AZs; automatic failover promotes a replica to primary

```text
Primary (AZ-a)  ←─── writes
    │  async replication
    ├──▶ Replica (AZ-b)  ←─── reads
    └──▶ Replica (AZ-c)  ←─── reads
```

### Cluster Mode
**Cluster Mode Disabled**: single shard (all data on one primary + replicas). Simple, up to 500 GB.

**Cluster Mode Enabled**: data **sharded** across up to 500 shards. Each shard has its own primary + replicas. Total storage scales to hundreds of TB. Use when your dataset exceeds single-node capacity or you need write scaling across shards.

```text
Cluster Mode Enabled:
Shard 1: Primary-1 + Replicas  (keys: 0–5460)
Shard 2: Primary-2 + Replicas  (keys: 5461–10922)
Shard 3: Primary-3 + Replicas  (keys: 10923–16383)
```

### Redis data structures for common use cases

| Use case | Redis data structure |
|---|---|
| Session store | Hash (field per session attribute) or String (serialised JSON) |
| Leaderboard / rankings | **Sorted Set** — score-ordered, O(log N) rank lookup |
| Rate limiting | String with INCR + EXPIRE |
| Pub/Sub messaging | Pub/Sub channels |
| Recent items / activity feed | List (LPUSH + LTRIM) |
| Unique visitor count | HyperLogLog (probabilistic, memory-efficient) |
| Geospatial queries | Geo commands (GEOADD, GEODIST, GEORADIUS) |

### Persistence options
- **RDB (Redis Database snapshots)**: point-in-time snapshots at intervals. Fast restore, small files, but data loss since last snapshot.
- **AOF (Append-Only File)**: logs every write operation. Near-zero data loss, but slower and larger files.
- **No persistence**: pure cache — fastest, loses all data on restart.

### Redis AUTH and encryption
- **AUTH token** (Redis AUTH): password required before issuing commands
- **In-transit encryption** (TLS): encrypt traffic between clients and Redis
- **At-rest encryption**: KMS-encrypted storage for RDB snapshots and AOF
- **IAM authentication** (Redis 7+): use IAM roles instead of passwords (newer clusters only)

## ElastiCache for Memcached

Memcached is simpler than Redis — a pure, multi-threaded, distributed cache.

- **Multi-threaded**: can utilise multiple CPU cores — better raw throughput than Redis for simple get/set at very high concurrency
- **No replication**: node loss = cache loss. Data is rebuilt from the source DB.
- **No persistence**: purely in-memory; all data is lost on node restart
- **Horizontal scaling**: add/remove nodes; data is automatically redistributed using consistent hashing
- **Auto Discovery**: client libraries auto-discover all nodes in a cluster

### When to use Memcached
- Pure, simple object caching where data loss is acceptable
- Need to scale out horizontally across many nodes
- Multi-threaded performance is a priority
- No need for persistence, replication, or complex data types

## Architecture Patterns

### DB query caching
```text
App checks ElastiCache with query fingerprint as key
  ├── HIT:  return cached result set
  └── MISS: execute DB query → store result in cache with TTL → return
```
Dramatically reduces RDS read load for repeated read-heavy queries.

### Session store (stateless app tier)
```text
User logs in → session written to ElastiCache
Request hits any app server → reads session from ElastiCache
App servers are stateless → easy horizontal scaling
```
Essential for Auto Scaling Groups — as instances scale in/out, no session data is lost.

### Real-time leaderboard (Redis Sorted Set)
```text
ZADD leaderboard 9500 "player:alice"   # add/update score
ZREVRANK leaderboard "player:alice"    # get rank (0-indexed from top)
ZREVRANGE leaderboard 0 9              # top 10 players
```
O(log N) operations — serves millions of ranking queries per second.

### ElastiCache vs RDS vs DynamoDB

| | ElastiCache | RDS/Aurora | DynamoDB |
|---|---|---|---|
| Storage | In-memory | Disk | SSD + in-memory buffer |
| Latency | Microseconds | Milliseconds | Single-digit ms |
| Persistence | Optional (Redis) | Always | Always |
| Query model | Key-value (+ Redis structures) | SQL | Key-value + Query |
| Use case | Cache layer, sessions, leaderboards | Relational data | NoSQL at scale |

## Working with ElastiCache via boto3 and redis-py

In [ ]:
import boto3

ec_client = boto3.client('elasticache', region_name='us-east-1')

In [ ]:
# Create a Redis replication group (Multi-AZ, 1 primary + 2 replicas)
ec_client.create_replication_group(
    ReplicationGroupId='prod-redis',
    Description='Production Redis cache',
    AutomaticFailoverEnabled=True,   # promote replica automatically on primary failure
    MultiAZEnabled=True,
    NumCacheClusters=3,              # 1 primary + 2 replicas
    CacheNodeType='cache.r7g.large',
    Engine='redis',
    EngineVersion='7.1',
    CacheSubnetGroupName='my-cache-subnet-group',
    SecurityGroupIds=['sg-0cache'],
    AtRestEncryptionEnabled=True,
    TransitEncryptionEnabled=True,
    AuthToken='strong-auth-token-32chars-min',
    SnapshotRetentionLimit=7,        # keep daily snapshots for 7 days
    SnapshotWindow='03:00-04:00',    # UTC
    Tags=[{'Key': 'Environment', 'Value': 'production'}]
)
print("Redis replication group creation initiated")

In [ ]:
# Connect to Redis and demonstrate common patterns (requires redis-py)
# pip install redis
import redis
import json
import time

r = redis.Redis(
    host='prod-redis.abc123.ng.0001.use1.cache.amazonaws.com',
    port=6379,
    ssl=True,
    password='strong-auth-token-32chars-min',
    decode_responses=True
)

# --- Lazy loading pattern ---
def get_user(user_id: str) -> dict:
    cache_key = f'user:{user_id}'
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)   # cache HIT
    # cache MISS — fetch from DB
    user = {'id': user_id, 'name': 'Alice', 'email': 'alice@example.com'}  # DB fetch
    r.setex(cache_key, 300, json.dumps(user))  # cache for 5 minutes
    return user

# --- Session store ---
def save_session(session_id: str, user_id: str, ttl_seconds: int = 1800):
    r.hset(f'session:{session_id}', mapping={
        'userId':    user_id,
        'createdAt': int(time.time())
    })
    r.expire(f'session:{session_id}', ttl_seconds)

# --- Rate limiting ---
def is_rate_limited(user_id: str, limit: int = 100, window: int = 60) -> bool:
    key = f'ratelimit:{user_id}:{int(time.time()) // window}'
    count = r.incr(key)
    if count == 1:
        r.expire(key, window)  # set expiry on first increment
    return count > limit

print("Redis patterns defined")

In [ ]:
# Sorted Set leaderboard
leaderboard = 'game:leaderboard'

# Add/update scores
r.zadd(leaderboard, {'player:alice': 9500, 'player:bob': 8200, 'player:carol': 9800})

# Get top 3 (highest scores)
top3 = r.zrevrange(leaderboard, 0, 2, withscores=True)
print("Top 3:", top3)

# Get rank of a specific player (0 = top)
rank = r.zrevrank(leaderboard, 'player:alice')
print(f"Alice rank: #{rank + 1}")

# Increment score (atomic)
r.zincrby(leaderboard, 200, 'player:alice')
print(f"Alice new score: {r.zscore(leaderboard, 'player:alice')}")

In [ ]:
# Create a Memcached cluster (simpler — no replication, no persistence)
ec_client.create_cache_cluster(
    CacheClusterId='dev-memcached',
    Engine='memcached',
    CacheNodeType='cache.t3.medium',
    NumCacheNodes=3,             # 3 nodes, client distributes keys across them
    CacheSubnetGroupName='my-cache-subnet-group',
    SecurityGroupIds=['sg-0cache'],
    Tags=[{'Key': 'Environment', 'Value': 'dev'}]
)
print("Memcached cluster creation initiated")

## Summary

| Concept | Key Takeaway |
|---|---|
| Lazy loading | Cache on first read; stale data possible; cache failure non-fatal |
| Write-through | Always fresh; write penalty; unused data fills cache |
| Session store | Stateless app tier; any server reads session from cache |
| Redis vs Memcached | Redis: persistence, replication, rich data types; Memcached: simple, multi-threaded |
| Exam default | Almost always Redis — Memcached only for simplicity/multi-threading |
| Replication Group | 1 primary + up to 5 replicas; Multi-AZ automatic failover |
| Cluster Mode Disabled | Single shard; simple; up to ~500 GB |
| Cluster Mode Enabled | Sharded across up to 500 shards; write scaling; large datasets |
| Sorted Set | Leaderboards, rankings; O(log N) operations |
| Redis persistence | RDB (snapshots) or AOF (every write); none = pure cache |
| Redis security | AUTH token + TLS in transit + KMS at rest |
| Memcached | No replication, no persistence; multi-threaded; pure cache |
| ElastiCache vs DAX | DAX = DynamoDB only, drop-in; ElastiCache = any backend, Redis/Memcached API |
| ElastiCache vs RDS | ElastiCache: microsecond, in-memory, cache layer; RDS: durable, SQL, milliseconds |